In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
pip install -q transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 92.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [3]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# EDA

In [6]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train_df.shape)
print(test_df.shape)

(2000, 8)
(500, 7)


In [7]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
train_df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [9]:
train_df.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

# Train-Test Split

In [10]:
train_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df["answer"], random_state=4524)

# Evaluation Metric

## MAP@3
Score = 1.0 if correct answer is rank 1, 0.5 if rank 2, 1/3 if rank 3, else 0.

In [11]:
def mean_average_precision_at_3(true_labels, top3_preds):
    scores = []
    for true_label, preds in zip(true_labels, top3_preds):
        if true_label == preds[0]:
            scores.append(1.0)
        elif true_label == preds[1]:
            scores.append(0.5)
        elif true_label == preds[2]:
            scores.append(1 / 3)
        else:
            scores.append(0.0)
    return np.mean(scores)

# Model 1: LSTM + Transformer

## Model configs

In [12]:
CONFIG_M1 = {
    "model_name": "scratch-lstm-transformer",
    "embed_dim": 128,
    "lstm_hidden": 128,
    "num_heads": 4,
    "num_layers": 2,
    "batch_size": 16,
    "lr": 1e-3,
    "epochs": 5,
    "max_length": 128,
    "seed": 4524
}

## Dataset Class


### 1. Core Purpose
The `MCQDataset` class converts raw multiple-choice question data into tokenized tensor representations for Transformer models.

For every single question, it pairs the main question prompt with each of the 5 answer choices ($A$ through $E$). This creates 5 distinct sequence pairs per question, allowing the model to evaluate all choices in parallel.

---

### 2. Sequence Pairing & Tokenization
When `prompt` and `option_text` are passed into `self.tokenizer(prompt, option_text)`:

* **Text Format:** Hugging Face automatically formats the sequence pair using special tokens:
  $$\text{[CLS] Question Prompt [SEP] Option Text [SEP]}$$
* **Segment Masking (`token_type_ids`):** Assigns a value of `0` to tokens belonging to the prompt and `1` to tokens belonging to the option, allowing the model to distinguish where the question ends and the choice begins.

---

### 3. Tensor Shape Transformations
Inside `_tokenize_options`, the tensor shape changes step-by-step:

1. **Tokenizer Output:** Shape $(1, \text{max\_length})$
   * When `return_tensors="pt"` is set, the tokenizer assumes a batch input by default, adding an outer batch dimension of size `1`.
2. **After `.squeeze(0)`:** Shape $(\text{max\_length},)$
   * Strips the temporary outer dimension of `1`, flattening the output into a 1D vector of token IDs for that option.
3. **After `torch.stack()`:** Shape $(5, \text{max\_length})$
   * Combines the 5 flattened option vectors ($A$ through $E$) into a single 2D matrix representing one full question.
4. **PyTorch `DataLoader` Output:** Shape $(\text{Batch\_Size}, 5, \text{max\_length})$
   * When batching (e.g., `batch_size = 16`), PyTorch stacks 16 individual question matrices into a 3D batch tensor for model training.

---

### 4. Data Retrieval Logic (`__getitem__`)
The `__getitem__(idx)` method handles row lookup and returns data based on the mode:

* **Training / Validation Mode (`has_labels = True`):**
  * Tokenizes the prompt paired with all 5 option choices into a $(5, \text{max\_length})$ tensor.
  * Converts the target text answer (e.g., `'B'`) into a numerical index (`0` to `4`) using `LABEL_TO_IDX`.
  * Returns `(input_ids, label)`.

* **Testing / Inference Mode (`has_labels = False`):**
  * Tokenizes the prompt paired with all 5 option choices into a $(5, \text{max\_length})$ tensor.
  * Retrieves the unique question identifier (`row["id"]`).
  * Returns `(input_ids, question_id)` so predictions can be mapped back to sample IDs in submission files.

In [13]:
# Map string choice labels to numerical target indices
LABEL_TO_IDX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

# List of column names containing the text for options A through E
OPTION_COLUMNS = ["A", "B", "C", "D", "E"]


class MCQDataset(Dataset):
  def __init__(self, records, tokenizer, max_length, has_labels=True):
    """Parameters:
    - records: List of dictionaries, where each dict is one question row.
    - tokenizer: Hugging Face PreTrainedTokenizer instance 
    - max_length: Maximum sequence length (in tokens) for padding/truncation.
    - has_labels: Set to True for training/validation, False for test set.
    """
    self.records = records
    self.tokenizer = tokenizer
    self.max_length = max_length
    self.has_labels = has_labels

  def __len__(self):
    return len(self.records)

  def _tokenize_options(self, prompt, options):
    """Helper method: Tokenizes the question prompt paired with each of the 5 options.

    Returns:- A stacked PyTorch tensor of shape (5, max_length) containing input IDs for all 5 options.
    """
    encoded_options = []

    for option_text in options:
      # Pass both prompt and option_text
      # Format: [CLS] prompt [SEP] option_text [SEP]
      tokens = self.tokenizer(
          prompt,
          option_text,
          truncation=True,  # Truncate if prompt + option exceeds max_length
          padding="max_length",  # Pad with zeros up to max_length
          max_length=self.max_length,
          return_tensors="pt", 
      )

      encoded_options.append(tokens["input_ids"].squeeze(0))

    # Turns a list of 5 tensors of shape (128,) into a single tensor of shape (5, 128).
    return torch.stack(encoded_options)

  def __getitem__(self, idx):
    """Fetches a single question by index, tokenizes all 5 options, and returns tensors.
    """
    row = self.records[idx]

    # Convert prompt and options to plain strings
    prompt = str(row["prompt"])
    options = [str(row[col]) for col in OPTION_COLUMNS]

    # Tokenize prompt paired with all 5 choices -> Shape: (5, max_length)
    input_ids = self._tokenize_options(prompt, options)

    if self.has_labels:
      # Training / Validation mode: return input_ids and integer ground-truth label
      label = LABEL_TO_IDX[row["answer"]]
      return input_ids, torch.tensor(label, dtype=torch.long)
    else:
      # Testing / Inference mode: return input_ids and question ID for tracking predictions
      return input_ids, row["id"]

## Model
Each option is encoded independently: Embedding → BiLSTM → Transformer Encoder → pooled → scored.

The 5 option scores become the logits for classification.

### 1. The Big Picture
The `HybridQAModel` evaluates multiple-choice options by processing the question prompt paired with each answer choice. Instead of relying on a single neural network layer, it combines **sequential processing** (LSTM) with **global self-attention** (Transformer) to capture both word order and long-range relationships before outputting a confidence score for each option.

---

### 2. Core Processing Engines: LSTM vs. Transformer

* **Bidirectional LSTM:** Reads text sequentially in both directions (left-to-right and right-to-left). It specializes in **local sentence structure**, grammar, and word order (e.g., recognizing how words like *"not"* alter nearby meaning).
* **Transformer Encoder:** Uses **Self-Attention** to look at all words simultaneously. It specializes in **global context**, directly linking keywords in the question prompt to relevant details in the answer choice regardless of distance.

---

### 3. Step-by-Step Layer Breakdown

1. **Embedding Layer (`nn.Embedding`)**
   * **Role:** Translates raw token IDs into dense vectors in a continuous concept space where semantically similar words sit close together.
   * **Padding Handling:** Uses `padding_idx=0` to keep `[PAD]` tokens zeroed out so they don't introduce artificial meaning.

2. **Bidirectional LSTM (`nn.LSTM`)**
   * **Role:** Reads through the embedded word sequence step-by-step from both directions, constructing a rich representation of local sentence structure and word ordering.

3. **Layer Normalization (`nn.LayerNorm`)**
   * **Role:** Standardizes and re-scales feature activations coming out of the LSTM.
   * **Why It Matters:** Keeps signals stable across training iterations so the Transformer receives balanced, non-exploding inputs.

4. **Transformer Encoder (`nn.TransformerEncoder`)**
   * **Role:** Allows tokens across the entire prompt-option pair to pay attention to one another.
   * **Padding Mask (`src_key_padding_mask`):** Tells self-attention heads to ignore empty padding slots so background zeros don't dilute key attention weights.

5. **Hybrid Feature Pooling (Mean + Max Pooling)**
   * **Role:** Compresses the full sequence of token vectors into a single summary vector per option choice.
   * **Mean Pooling:** Averages features across all tokens to capture the overall topic and general background context.
   * **Max Pooling:** Extracts the highest feature activations to preserve sharp, prominent highlights (such as specific entities or key terms).
   * **Concatenation (`torch.cat`):** Fuses broad context and key highlights into one complete summary vector.

6. **Linear Classifier (`nn.Linear`)**
   * **Role:** Evaluates the fused summary vector for each option and produces a single numerical score (logit). The choice with the highest score represents the model's predicted answer.

In [14]:
class HybridQAModel(nn.Module):
    
  def __init__(self, vocab_size, embed_dim, lstm_hidden, num_heads, num_layers):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
      
    self.lstm = nn.LSTM(embed_dim, lstm_hidden, batch_first=True, bidirectional=True)
    self.layer_norm = nn.LayerNorm(lstm_hidden * 2)
    
    encoder_layer = nn.TransformerEncoderLayer(d_model=lstm_hidden * 2, nhead=num_heads, batch_first=True)
    self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
    
    self.classifier = nn.Linear(lstm_hidden * 4, 1)

  def forward(self, input_ids):
    batch_size, num_options, seq_len = input_ids.shape
    flat_ids = input_ids.view(batch_size * num_options, seq_len)
    padding_mask = flat_ids == 0
      
    embedded = self.embedding(flat_ids)

    lstm_out, _ = self.lstm(embedded)
    lstm_out = self.layer_norm(lstm_out)

    encoded = self.transformer(lstm_out, src_key_padding_mask=padding_mask)

    avg_pool = encoded.mean(dim=1)
    max_pool = encoded.max(dim=1)[0]
    pooled = torch.cat([avg_pool, max_pool], dim=1)

    logits = self.classifier(pooled).view(batch_size, num_options)

    return logits

## Training Helpers
- Run one epoch of training
- Run one epoch of evaluation
- The outer loop

In [15]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for input_ids, labels in loader:
        input_ids, labels = input_ids.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_top3 = [], []

    with torch.no_grad():
        for input_ids, labels in loader:
            input_ids, labels = input_ids.to(device), labels.to(device)
            logits = model(input_ids)
            total_loss += criterion(logits, labels).item()

            top3 = torch.argsort(logits, dim=1, descending=True)[:, :3]
            all_labels.extend(labels.cpu().tolist())
            all_top3.extend(top3.cpu().tolist())

    val_loss = total_loss / len(loader)
    val_map3 = mean_average_precision_at_3(all_labels, all_top3)
    val_acc = np.mean([label == preds[0] for label, preds in zip(all_labels, all_top3)])
    return val_loss, val_acc, val_map3

In [16]:
def train_model(model, train_loader, val_loader, config, checkpoint_path):
    run_name = "Scratch-LSTM-Transformer-fix"
    wandb.init(project="24f3004524-t22026", name=run_name, config=config)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    best_map3 = 0.0

    for epoch in range(config["epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_map3 = evaluate(model, val_loader, criterion)

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "val_map3": val_map3,
        })
        print(f"Epoch {epoch+1} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_map3={val_map3:.4f}")

        torch.save(model.state_dict(), checkpoint_path)

    wandb.finish()
    return best_map3

## Build Datasets & Run Training

In [17]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size

train_dataset = MCQDataset(train_df.to_dict("records"), tokenizer, CONFIG_M1["max_length"])
val_dataset = MCQDataset(val_df.to_dict("records"), tokenizer, CONFIG_M1["max_length"])

train_loader = DataLoader(train_dataset, batch_size=CONFIG_M1["batch_size"], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG_M1["batch_size"], shuffle=False)

model_m1 = HybridQAModel(
    vocab_size=vocab_size,
    embed_dim=CONFIG_M1["embed_dim"],
    lstm_hidden=CONFIG_M1["lstm_hidden"],
    num_heads=CONFIG_M1["num_heads"],
    num_layers=CONFIG_M1["num_layers"],
).to(device)

train_model(model_m1, train_loader, val_loader, CONFIG_M1, checkpoint_path="model1_best.pt")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260730_124702-zipmc75w
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run Scratch-LSTM-Transformer-fix
wandb: ⭐️ View project at https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026
wandb: 🚀 View run at https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026/runs/zipmc75w
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1 | train_loss=0.8275 | val_loss=0.5358 | val_map3=0.9975
Epoch 2 | train_loss=0.4561 | val_loss=0.4447 | val_map3=1.0000
Epoch 3 | train_loss=0.4207 | val_loss=0.4430 | val_map3=0.9946
Epoch 4 | train_loss=0.4072 | val_loss=0.4308 | val_map3=1.0000


wandb: updating run metadata


Epoch 5 | train_loss=0.4026 | val_loss=0.4244 | val_map3=1.0000


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 4-4, summary, console lines 6-6
wandb: 
wandb: Run history:
wandb:        epoch ▁▃▅▆█
wandb:   train_loss █▂▁▁▁
wandb: val_accuracy ▆█▁██
wandb:     val_loss █▂▂▁▁
wandb:     val_map3 ▅█▁██
wandb: 
wandb: Run summary:
wandb:        epoch 5
wandb:   train_loss 0.40265
wandb: val_accuracy 1
wandb:     val_loss 0.42443
wandb:     val_map3 1
wandb: 
wandb: 🚀 View run Scratch-LSTM-Transformer-fix at: https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026/runs/zipmc75w
wandb: ⭐️ View project at: https://wandb.ai/pranay12aggarwal-indian-institute-of-technology-madras/24f3004524-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260730_124702-zipmc75w/logs


0.0

# Predict on Test Set

In [18]:
def predict_top3(model, loader, idx_to_label):
    model.eval()
    ids, predictions = [], []
    with torch.no_grad():
        for input_ids, row_ids in loader:
            input_ids = input_ids.to(device)
            logits = model(input_ids)
            top3 = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().numpy()

            for row_id, pred_idxs in zip(row_ids, top3):
                letters = " ".join(idx_to_label[i] for i in pred_idxs)
                predictions.append(letters)
                ids.append(row_id.item() if isinstance(row_id, torch.Tensor) else row_id)
    return ids, predictions


idx_to_label = {v: k for k, v in LABEL_TO_IDX.items()}

test_dataset = MCQDataset(test_df.to_dict("records"), tokenizer, CONFIG_M1["max_length"], has_labels=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG_M1["batch_size"], shuffle=False)

best_model_m1 = HybridQAModel(
    vocab_size=vocab_size,
    embed_dim=CONFIG_M1["embed_dim"],
    lstm_hidden=CONFIG_M1["lstm_hidden"],
    num_heads=CONFIG_M1["num_heads"],
    num_layers=CONFIG_M1["num_layers"],
).to(device)
best_model_m1.load_state_dict(torch.load("model1_best.pt"))

test_ids, test_predictions = predict_top3(best_model_m1, test_loader, idx_to_label)

# Submission Cell

In [19]:
submission = pd.DataFrame({"ID": test_ids, "Prediction": test_predictions})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,A E B
1,2,B E A
2,3,B A D
3,4,E D C
4,5,C D A
